# Day 12 · 时间数据基础(W2 第 5 天)

目标:解析日期、按时间索引切片、重采样(resample)、滚动窗口(rolling)。

今日节奏:40min 学习 + 15min 动手 + 5min 自检。

> 打开方式:JupyterLab 文件树里进 `ai-learning/练习/` 双击本文件,逐格 Shift+Enter。


In [ ]:
import sys
import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)
print("环境 OK!开始今天的练习 →")

rng = np.random.default_rng(11)
dates = pd.date_range("2026-01-01", periods=84, freq="D")
values = 100 + 0.5 * np.arange(84) + rng.normal(0, 12, 84) + 10 * np.sin(np.arange(84) / 7)
ts = pd.DataFrame({"date": dates, "value": values.round(1)})
print(ts.head())


## 任务 1:解析日期 + .dt 属性

- `pd.to_datetime(列)` → 把字符串列解析成 datetime 类型
- 解析后可用 `.dt.year / .dt.month / .dt.dayofweek`(0=周一)等


In [ ]:
ts["date"] = pd.to_datetime(ts["date"])
print("类型:", ts["date"].dtype)
print("涉及的年份:", ts["date"].dt.year.unique().tolist())
print("星期分布:\n", ts["date"].dt.dayofweek.value_counts().sort_index())


## 任务 2:日期索引与切片

`set_index("date")` 后,DataFrame 就变成时间序列,可以:

- `idx.loc["2026-02"]` → 直接按"年-月"切
- `idx.loc["2026-01-10":"2026-01-20"]` → 按日期范围切


In [ ]:
idx = ts.set_index("date")
print("2026 年 2 月的记录数:", idx.loc["2026-02"].shape[0])
print(idx.loc["2026-01-10":"2026-01-14"])


## 任务 3:resample 重采样

把细粒度数据聚合成粗粒度:

- `resample("W").sum()` → 每周合计(周日为界;可用 "W-MON" 指定周一)
- `resample("ME").mean()` → 每月平均(旧写法 "M" 已弃用,用 "ME")

记法:频率右对齐,`"2W"` 就是每两周。


In [ ]:
weekly = idx["value"].resample("W").sum()
monthly = idx["value"].resample("ME").mean()
print("每周合计:\n", weekly)
print("每月平均:\n", monthly.round(1))


## 任务 4:rolling 滚动窗口

`rolling(7).mean()` 每行取"自己和前面 6 天"的平均,用来平滑噪声、看趋势。

- `.dropna()` 丢掉前 6 个没有完整窗口的行
- 股票里常用的 20 日均线就是这个


In [ ]:
smooth = idx["value"].rolling(7).mean()
print("滚动平均(前 10 行):\n", smooth.dropna().head().round(1))
print("波动比原始数据小得多,这就是平滑的意义")


## 任务 5:小挑战 🔥(一图看懂 12 周)

1. 左图:每周合计的柱状图
2. 右图:每日值 vs 7 日滚动平均


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3.5))

plt.subplot(1, 2, 1)
weekly.plot(kind="bar", color="steelblue")
plt.title("Weekly total")
plt.ylabel("Value")

plt.subplot(1, 2, 2)
plt.plot(idx["value"], alpha=0.4, label="daily")
plt.plot(smooth, label="7-day avg", linewidth=2)
plt.legend()
plt.title("Daily vs rolling average")

plt.tight_layout()
plt.show()


## 自检清单(5 问,答不上就回看今天的格子)

1. 怎么把字符串日期转成 datetime? → `pd.to_datetime(列)`
2. 按"年-月"切片的前提? → 先把日期列 set_index
3. resample 干什么? → 按固定频率把细粒度聚合成粗粒度
4. rolling(7).mean() 是什么? → 7 日移动平均,平滑噪声
5. "M" 还能用吗? → 已弃用,用 "ME"(月末频率)

## 📝 收盘动作

```powershell
cd D:\01_Study\ai-learning
git add -A; git commit -m "day12: time series"; git push
```

然后跟助手说"生成日志"。
